# Quantum Canary: package demo

**Quantum Canary** recovers the parameters that decide whether a qubit is fit for computation from a fixed set of **17 short probe circuits per qubit** (11,900 shots):

| Probe | Circuits | Recovers |
|---|---|---|
| T1 decay | 3 | **T1**: energy relaxation time |
| Ramsey X/Y | 8 (4 delays × 2 quadratures) | **Δω**: qubit detuning (phase unwrapped across all delays) and **T2\***: Ramsey dephasing time |
| Hahn echo | 3 | **T2**: echo dephasing time (the T2 IBM reports) |
| Gate repetition | 3 (frame-twirled native G·G⁻¹ pairs) | **ε_sx**: incoherent single-qubit gate error |

Every estimate has a 1σ uncertainty and the reduced χ² of its fit, and every qubit gets a health flag (GREEN / YELLOW / RED).

**Hardware:** superconducting qubits on IBM Quantum (Qiskit Runtime). **Simulation:** superconducting and trapped-ion qubits; trapped ions are supported in simulation only.

**Contents**
1. Install
2. Quick start in simulation
3. Working with the results
4. Settings
5. Trapped ions and the ideal regime
6. The building blocks: priors, probe circuits, inversion
7. The simulator
8. A hardware dry run on a fake IBM backend (no account needed)
9. Real IBM hardware
10. Online monitoring inside a running workflow
11. Command line

## 1. Install

```bash
pip install quantum-canary            # engine, simulator, online monitoring
pip install "quantum-canary[ibm]"     # + qiskit-ibm-runtime, to run on IBM hardware (sections 8-9)
```

From a clone of the repository: `pip install -e ".[ibm]"`. Python 3.10 or newer.

Uncomment the next cell to install from inside this notebook.

In [1]:
# %pip install "quantum-canary[ibm]" 

In [2]:
import warnings
warnings.filterwarnings("ignore")          # quiet Qiskit/fit warnings in the demo

import numpy as np
import quantum_canary as canary          # not "qc": that is Qiskit's conventional name for a QuantumCircuit

print("quantum-canary", canary.__version__)

quantum-canary 1.0.0


## 2. Quick start in simulation

`run_simulation()` builds simulated qubits with **hidden true parameters**, runs the Canary probe on them, inverts the counts and keeps the truth for comparison.

With `regime="nisq"` (the default) every simulated qubit has drifting T1 and frequency, slow shot-to-shot frequency noise, a residual detuning of up to ±15 kHz that also acts during gate pulses, coherent gate over-rotation, readout error, and a calibration record that is about **1 hour stale** (±25% of the truth for superconducting qubits). Canary only ever sees the counts and that stale record.

In [3]:
report = canary.run_simulation("superconducting", n_qubits=2, regime="nisq")
report.print()

━━ Quantum Canary v1.0.0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Backend     simulator · superconducting · in-silico simulation · nisq regime
  Qubits      0, 1
  Probe       17 circuits/qubit · 11,900 shots/qubit · 10 interleaved blocks
              per circuit: T1 300 · Ramsey 1000 · gate-rep 500 · echo 500 shots
  Time        2026-10-03 15:32:09 UTC
              True values are the simulator's hidden parameters (T2 = echo T2; ε excludes detuning).

── Qubit 0  ● GREEN ──────────────────────────────────────────────────────────
  Priors      SIMULATED CALIBRATION RECORD  (1 h old, ±25% of truth)
              T1 164.8 µs · T2 107 µs
              SPAM  p0|1 0.0315 · p1|0 0.0072

  Parameter     Canary estimate             Unit   χ²/dof         True    Error
  T1            285 ± 16                    µs       0.18        279.8    +1.9%
  T2 (echo)     90.2 ± 6.7                  µs       0.23        92.33    -2.3%
  T2* (Ramsey)  74.0 ± 4.0                  µs       

Each qubit block shows where the **priors** came from (here a simulated stale calibration record; on hardware **LIVE** backend calibration or **ARCHITECTURE DEFAULTS**), the priors themselves, every recovered parameter with its uncertainty and χ²/dof, and, in simulation, the true value and the error.

`report.print()` writes the report to the screen; `report.render()` returns it as a string.

## 3. Working with the results

`report.qubits` is a list of `QubitReport`s. Each one has:

- `result`: the engine's `InversionResult` (all estimates, sigmas and χ²/dof)
- `profile`: the `BackendProfile` the probe was built from (priors and their provenance)
- `health`: `"GREEN"`, `"YELLOW"` or `"RED"`
- `truth`: the true parameters (simulation only)

In [4]:
q = report.qubits[0]
r = q.result
print("health      :", q.health)
print("T1          :", r.T1_s, "+/-", r.T1_sigma_s, "s")
print("T2 (echo)   :", r.T2_s, "+/-", r.T2_sigma_s, "s")
print("T2* (Ramsey):", r.T2_ramsey_s, "+/-", r.T2_ramsey_sigma_s, "s")
print("delta_omega :", r.delta_omega, "+/-", r.delta_omega_sigma, "rad/s")
print("epsilon_sx  :", r.epsilon_sx, "+/-", r.epsilon_sx_sigma)
print("chi2/dof    : T1", round(r.t1_chi2_dof, 2), " Ramsey", round(r.ramsey_chi2_dof, 2),
      " echo", round(r.echo_chi2_dof, 2), " gate", round(r.gate_chi2_dof, 2))
print("prior       :", q.profile.prior_confidence, "T1_prior", q.profile.T1_prior_s, "T2_prior", q.profile.T2_prior_s)
print("truth       :", {k: q.truth[k] for k in ("T1_s", "T2_s", "delta_omega", "epsilon_sx")})

health      : GREEN
T1          : 0.0002849979149955478 +/- 1.5565069186056645e-05 s
T2 (echo)   : 9.019548380473535e-05 +/- 6.678306902900148e-06 s
T2* (Ramsey): 7.400580484039575e-05 +/- 3.990777369776681e-06 s
delta_omega : -33566.42174233379 +/- 795.4704290667238 rad/s
epsilon_sx  : 0.0002425699155986096 +/- 1.6136125083526354e-05
chi2/dof    : T1 0.18  Ramsey 0.7  echo 0.23  gate 1.86
prior       : prior_informed T1_prior 0.00016482923595093575 T2_prior 0.0001069790346387992
truth       : {'T1_s': 0.00027975683848734817, 'T2_s': 9.233242862101364e-05, 'delta_omega': -34065.002149578664, 'epsilon_sx': 0.000282010750429329}


A table across qubits (needs `pandas`), and the full report as JSON:

In [5]:
import pandas as pd

table = pd.DataFrame([{
    "qubit": q.qubit, "health": q.health,
    "T1_us": q.result.T1_s * 1e6, "T1_true_us": q.truth["T1_s"] * 1e6,
    "T2_us": q.result.T2_s * 1e6, "T2_true_us": q.truth["T2_s"] * 1e6,
    "dw_kHz": q.result.delta_omega / (2 * np.pi * 1e3),
    "dw_true_kHz": q.truth["delta_omega"] / (2 * np.pi * 1e3),
    "eps": q.result.epsilon_sx, "eps_true": q.truth["epsilon_sx"],
} for q in report.qubits])
table.round(4)

,qubit,health,T1_us,T1_true_us,T2_us,T2_true_us,dw_kHz,dw_true_kHz,eps,eps_true
0,0,GREEN,284.9979,279.7568,90.1955,92.3324,-5.3423,-5.4216,0.0002,0.0003
1,1,YELLOW,131.8148,130.6010,94.9034,103.9803,7.8358,7.9009,0.0004,0.0005


In [6]:
import json

data = report.to_dict()                    # JSON-serialisable dict
with open("canary_report.json", "w") as f:
    json.dump(data, f, indent=2, default=str)
print(list(data), "->", "canary_report.json")

['mode', 'backend', 'architecture', 'regime', 'job_id', 'shots', 'probe_blocks', 'circuits_per_qubit', 'timestamp', 'qubits'] -> canary_report.json


## 4. Settings

| Argument | Default | Meaning |
|---|---|---|
| `architecture` | `"superconducting"` | or `"trapped_ion"` (simulation only) |
| `n_qubits` | `3` | number of simulated qubits |
| `regime` | `"nisq"` | `"nisq"`: drift, dephasing, detuning, coherent error, stale priors; `"ideal"`: noise exactly as Canary's model assumes |
| `seed` | benchmark offset | seed of qubit 0; qubit i uses `seed + i` |
| `shots` | `canary.DEFAULT_SHOTS` | per-circuit shots per probe family; pass only the ones to change |
| `probe_blocks` | `10` | the probe list runs in this many interleaved blocks so every probe samples the same time window; `1` runs each circuit's shots in one go |

`run_ibm()` takes the same `shots` and `probe_blocks`.

In [7]:
print("default shots per circuit:", canary.DEFAULT_SHOTS)
print("default probe blocks     :", canary.DEFAULT_PROBE_BLOCKS)

custom = canary.run_simulation("superconducting", n_qubits=1, regime="nisq", seed=42,
                           shots={"ramsey": 2000, "echo": 1000})
print("shots per qubit with more Ramsey/echo shots:", custom.shots_per_qubit)
custom.print()

default shots per circuit: {'t1': 300, 'ramsey': 1000, 'gate': 500, 'echo': 500}
default probe blocks     : 10


shots per qubit with more Ramsey/echo shots: 21400
━━ Quantum Canary v1.0.0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Backend     simulator · superconducting · in-silico simulation · nisq regime
  Qubits      0
  Probe       17 circuits/qubit · 21,400 shots/qubit · 10 interleaved blocks
              per circuit: T1 300 · Ramsey 2000 · gate-rep 500 · echo 1000 shots
  Time        2026-10-03 15:32:32 UTC
              True values are the simulator's hidden parameters (T2 = echo T2; ε excludes detuning).

── Qubit 0  ● YELLOW ─────────────────────────────────────────────────────────
  Priors      SIMULATED CALIBRATION RECORD  (1 h old, ±25% of truth)
              T1 116.4 µs · T2 59.19 µs
              SPAM  p0|1 0.0253 · p1|0 0.0079

  Parameter     Canary estimate             Unit   χ²/dof         True    Error
  T1            95.7 ± 4.4                  µs       0.21        104.6    -8.6%
  T2 (echo)     57.4 ± 2.9                  µs       0.86        57.45    -0.0%
  

## 5. Trapped ions and the ideal regime

The same call simulates trapped-ion qubits; times are shown in seconds or milliseconds and the detuning in Hz. `regime="ideal"` is the correctly specified case: the simulated noise is exactly what Canary's model assumes.

In [8]:
ions = canary.run_simulation("trapped_ion", n_qubits=1, regime="ideal")
ions.print()

━━ Quantum Canary v1.0.0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Backend     simulator · trapped_ion · in-silico simulation · ideal regime
  Qubits      0
  Probe       17 circuits/qubit · 11,900 shots/qubit · circuit-by-circuit
              per circuit: T1 300 · Ramsey 1000 · gate-rep 500 · echo 500 shots
  Time        2026-10-03 15:32:33 UTC
              True values are the simulator's hidden parameters (T2 = echo T2; ε excludes detuning).

── Qubit 0  ● GREEN ──────────────────────────────────────────────────────────
  Priors      SIMULATED CALIBRATION RECORD  (fresh, ±5% of truth)
              T1 124.7 s · T2 512.3 ms
              SPAM  p0|1 0.0059 · p1|0 0.0041

  Parameter     Canary estimate             Unit   χ²/dof         True    Error
  T1            132.7 ± 6.2                 s        0.65        130.4    +1.7%
  T2 (echo)     449 ± 32                    ms       0.84        497.9    -9.8%
  T2* (Ramsey)  506 ± 25                    ms       0.10       

## 6. The building blocks: priors, probe circuits, inversion

`run_simulation()` and `run_ibm()` are thin wrappers around three steps of the engine (`quantum_canary.inversion`):

1. a **`BackendProfile`**: the T1/T2/gate-error/readout priors that place the probe delays
2. **`build_probe_circuits(profile)`**: the 17 one-qubit circuits and their metadata
3. **`lindblad_inversion(counts, meta, profile, ...)`**: the counts → T1, T2, T2*, Δω, ε

Profiles can come from a backend's live calibration (`BackendProfile.from_ibm_backend(backend, qubit=q)`), from architecture defaults (`BackendProfile.from_architecture(...)`), or from your own priors:

In [9]:
default_profile = canary.BackendProfile.from_architecture("superconducting")
print("architecture-default priors: T1", default_profile.T1_prior_s, "T2", default_profile.T2_prior_s,
      "| confidence:", default_profile.prior_confidence)

my_profile = canary.BackendProfile(
    architecture="superconducting",
    T1_prior_s=150e-6, T2_prior_s=90e-6,                  # your own estimates
    dt_ns=canary.ARCH_DEFAULTS["superconducting"]["dt_ns"],
    prior_confidence="prior_informed")

circuits, meta = canary.build_probe_circuits(my_profile)
print(len(circuits), "circuits:", [c.name for c in circuits])
print("Ramsey delays (s):", np.round(meta["ramsey_delays_s"], 9))
print("gate-repetition N:", meta["gate_rep_N"])

architecture-default priors: T1 0.0002 T2 0.00014 | confidence: arch_default


17 circuits: ['t1_0', 't1_1', 't1_2', 'ramsey_X_0', 'ramsey_Y_0', 'ramsey_X_1', 'ramsey_Y_1', 'ramsey_X_2', 'ramsey_Y_2', 'ramsey_X_3', 'ramsey_Y_3', 'gate_rep_N208', 'gate_rep_N416', 'gate_rep_N833', 'echo_0', 'echo_1', 'echo_2']
Ramsey delays (s): [9.00e-06 4.50e-05 9.00e-05 1.35e-04]
gate-repetition N: [208, 416, 833]


In [10]:
circuits[3].draw("text")       # ramsey_X_0: H, delay, H, measure

┌───┐┌──────────────────┐┌───┐┌─┐
  q: ┤ H ├┤ Delay(40504[dt]) ├┤ H ├┤M├
     └───┘└──────────────────┘└───┘└╥┘
c: 1/═══════════════════════════════╩═
                                    0

Running the circuits anywhere and inverting the counts. Here they run on the package's simulator; on hardware they would go to a sampler (section 9 does this for you). Counts are one dict per circuit, `{"0": n0, "1": n1}`, in circuit order, with the per-family shot counts the inversion is told about.

In [11]:
sim = canary.simulation

truth = sim.generate_device("superconducting", n_qubits=1, seed=7, regime="ideal")
shots = canary.DEFAULT_SHOTS
shots_list = ([shots["t1"]] * meta["n_t1"] + [shots["ramsey"]] * meta["n_ramsey"]
              + [shots["gate"]] * meta["n_gate"] + [shots["echo"]] * meta["n_echo"])
counts = sim.simulate_probe(circuits, truth, shots=shots_list, dt_ns=my_profile.dt_ns, seed=1).counts_list
print("first counts:", counts[:3])

result = canary.lindblad_inversion(counts, meta, my_profile,
                               shots_t1=shots["t1"], shots_ramsey=shots["ramsey"],
                               shots_gate=shots["gate"], shots_echo=shots["echo"])
print(result.summary())

tv = sim.evaluate_truth_at(truth, 0, 0.0)
print("truth: T1 %.1f us  T2 %.1f us  dw/2pi %.3f kHz  eps %.2e" % (
    tv["T1_s"] * 1e6, tv["T2_s"] * 1e6, tv["delta_omega_rad_s"] / (2 * np.pi * 1e3),
    sim.effective_gate_epsilon(truth, 0, 0.0)))

first counts: [{'0': 60, '1': 240}, {'0': 112, '1': 188}, {'1': 141, '0': 159}]
Qubit 0 | T1=330.88±19.72µs  T2=97.68±15.20µs  Δω=6.082kHz  ε=5.12e-04
truth: T1 334.3 us  T2 92.3 us  dw/2pi 6.164 kHz  eps 4.98e-04


## 7. The simulator

`quantum_canary.simulation` is the device simulator Canary is validated on. Every numerical default is registered with its literature source and evidence class (A = measured, B = derived, C = assumption) in `MODEL_PROVENANCE`:

In [12]:
entry = sim.get_parameter_provenance("sc.quasi_static.t2_ramsey_over_echo")
print("value   :", entry["value"])
print("class   :", entry["evidence_class"])
print("source  :", entry["source"][:120], "...")
print(len(sim.MODEL_PROVENANCE), "registered parameters")

value   : (0.6405228758169934, 0.9090909090909091, 0.618421052631579, 0.3673469387755102, 0.7735849056603774, 0.7013888888888888, 0.7816091954022989, 0.7464788732394366, 0.7435897435897436, 0.5223880597014925)
class   : B
source  : Place, Rodgers, Mundada, ..., de Leon, Houck, 'New material platform for superconducting transmon qubits with coherence  ...
53 registered parameters


The simulation layer has its own validation suite (about two minutes): `canary.simulation._run_validation_tests()`, or `quantum-canary selftest` on the command line.

## 8. A hardware dry run on a fake IBM backend (no account needed)

`run_ibm()` is the hardware path: it reads each qubit's **live calibration** from the backend, builds and transpiles the probe circuits onto the requested physical qubits, submits **one** `SamplerV2` job, and inverts each qubit.

`qiskit-ibm-runtime` ships fake backends with real device calibration and a matching noise model, so the whole path can run locally (needs `quantum-canary[ibm]`; about a minute per qubit).

In [13]:
from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2

fake_backend = FakeProviderForBackendV2().backend("fake_fez")    # 156-qubit Heron calibration + noise
dry_run = canary.run_ibm(fake_backend, qubits=[13])
dry_run.print()

━━ Quantum Canary v1.0.0 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Backend     fake_fez · superconducting · fake IBM backend · local dry run (simulated noise model)
  Job         345bffc4-3e46-412f-a653-b93e8ef00ccd
  Qubits      13
  Probe       17 circuits/qubit · 11,900 shots/qubit · 10 interleaved blocks
              per circuit: T1 300 · Ramsey 1000 · gate-rep 500 · echo 500 shots
  Time        2026-10-03 15:34:54 UTC

── Qubit 13  ● GREEN ─────────────────────────────────────────────────────────
  Priors      LIVE  (read from the backend's calibration)
              sources: T1 live · T2 live · ε live · SPAM live
              T1 262.3 µs · T2 204.6 µs
              SPAM  p0|1 0.0161 · p1|0 0.0049

  Parameter     Canary estimate             Unit   χ²/dof
  T1            253 ± 12                    µs       1.45
  T2 (echo)     248 ± 17                    µs       0.59
  T2* (Ramsey)  207.7 ± 9.9                 µs       0.71
  Δω/2π         +0.060 ± 0.043         

## 9. Real IBM hardware

1. Save your IBM Quantum account **once** (the token is stored on your machine, not in the notebook).
2. Pick a backend and the physical qubits to check.
3. Run. All circuits go out as a single job; the report shows the job ID.

The cell is disabled by default: set `RUN_ON_IBM_HARDWARE = True` to submit a real job (it uses your IBM Quantum time).

In [14]:
RUN_ON_IBM_HARDWARE = False

if RUN_ON_IBM_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService

    # Once per machine:
    # QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="<YOUR_TOKEN>", overwrite=True)

    service = QiskitRuntimeService()
    backend = service.backend("ibm_fez")      # or: service.least_busy(operational=True, simulator=False)
    hw = canary.run_ibm(backend, qubits=[13, 14, 2])
    hw.print()
    with open("canary_ibm_report.json", "w") as f:
        json.dump(hw.to_dict(), f, indent=2, default=str)
else:
    print("Hardware run disabled. Set RUN_ON_IBM_HARDWARE = True to submit a job.")

Hardware run disabled. Set RUN_ON_IBM_HARDWARE = True to submit a job.


**How to read the hardware report**

- **Priors: LIVE** means T1, T2, gate error and readout error were read from the backend's calibration and the delays were placed around them. **ARCHITECTURE DEFAULTS** means the backend's T1/T2 were unavailable and the delays were spread over the plausible range.
- **GREEN**: every probe fits its model and T2* agrees with T2. **YELLOW**: a probe's χ²/dof exceeds its limit, or T2* and T2 differ by more than 20%: non-Markovian or coherent noise, or stale priors. **RED**: a fit failed to converge.
- A large |Δω| is the qubit's residual frequency offset from its drive at job time: something the backend's calibration record does not report.

## 10. Online monitoring inside a running workflow

`CanaryInterleaver` weaves a full Canary round into any list of algorithm circuits (VQE, QAOA, ...) every `interval_n` circuits, splits the results back apart, inverts every round, and tracks health and **drift against a baseline** across rounds. Your algorithm's circuits and results are untouched.

The demo below uses the simulator in place of a backend: the algorithm circuits run on a plain Aer simulator, the Canary circuits on two simulated qubits.

In [15]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Two monitored qubits with hidden truths and their (stale) calibration records.
truths = {q: sim.generate_device("superconducting", 1, seed=200 + q, regime="ideal") for q in (0, 1)}
profiles = {}
for q, tr in truths.items():
    prior = sim.generate_calibration_prior(tr)
    profiles[q] = canary.BackendProfile(architecture="superconducting",
                                    T1_prior_s=prior.T1_prior_s, T2_prior_s=prior.T2_prior_s,
                                    dt_ns=canary.ARCH_DEFAULTS["superconducting"]["dt_ns"],
                                    prior_confidence="prior_informed")

# Your algorithm's circuits (here: 20 trivial ones).
algo = []
for i in range(20):
    circ = QuantumCircuit(1, 1, name=f"algo_{i}"); circ.h(0); circ.measure(0, 0)
    algo.append(circ)

monitor = canary.CanaryInterleaver(qubit_ids=[0, 1], profiles=profiles, interval_n=10)
job_circuits, round_map = monitor.prepare(algo)    # submit job_circuits as one job

[Canary] Prepared job: 20 algo circuits + 2 Canary rounds × 34 circuits = 88 circuits total. ~340.0% overhead.


The overhead printed above is large only because this demo has 20 tiny algorithm circuits: each round adds 17 circuits per monitored qubit, so the overhead falls as `interval_n` (algorithm circuits between rounds) grows.

In [16]:
# Stand-in for running job_circuits on a backend: one counts dict per circuit, in order.
aer = AerSimulator()
results = [None] * len(job_circuits)
for rmap in round_map:
    for i in rmap["algo_indices"]:
        results[i] = aer.run(job_circuits[i], shots=1000).result().get_counts()
    for qid, meta_q in monitor.canary_metadata.items():
        start = rmap["canary_start_idx"] + meta_q["circuit_start_idx"]
        block = job_circuits[start:start + meta_q["n_circuits"]]
        shots_q = ([shots["t1"]] * meta_q["n_t1"] + [shots["ramsey"]] * meta_q["n_ramsey"]
                   + [shots["gate"]] * meta_q["n_gate"] + [shots["echo"]] * meta_q["n_echo"])
        counts_q = sim.simulate_probe(block, truths[qid], shots=shots_q,
                                      dt_ns=profiles[qid].dt_ns, seed=1000 * rmap["round_index"] + qid).counts_list
        results[start:start + meta_q["n_circuits"]] = counts_q

algo_results, rounds = monitor.process(results, round_map)

[Canary] Round 0: Canary Round 0 | 2026-10-03T15:35:00.987811+00:00
  Q0: GREEN T1=244.90µs T2=123.76µs ε=7.72e-04
  Q1: GREEN T1=80.10µs T2=49.47µs ε=9.17e-04
[Canary] Round 1: Canary Round 1 | 2026-10-03T15:35:01.145278+00:00
  Q0: GREEN T1=231.97µs T2=102.58µs ε=7.22e-04
  Q1: GREEN T1=77.39µs T2=44.61µs ε=8.59e-04


In [17]:
print(rounds[-1].summary_table())
print()
print("health of the last round:", rounds[-1].health_summary())
print("drift of qubit 0 vs its baseline round:", monitor.drift_report(0)["drifted"])
print("T1 trajectory of qubit 0 (s):", monitor.trajectory(0, "T1_s")[1])
print("algorithm results returned untouched:", len(algo_results[0]), "+", len(algo_results[1]), "counts dicts")

Canary Round 1 | 2026-10-03T15:35:01.145278+00:00
Qubit T1            T2            Δω (kHz)    ε_sx       Health  
0     231.97±10.95µs102.58±7.47µs 2.72±0.07   7.22e-04   GREEN   
1     77.39±3.68µs  44.61±3.18µs  6.13±0.19   8.59e-04   GREEN   

health of the last round: {0: 'GREEN', 1: 'GREEN'}
drift of qubit 0 vs its baseline round: {'T1_s': False, 'T2_s': False, 'delta_omega': False, 'epsilon_sx': False}
T1 trajectory of qubit 0 (s): [0.0002448999910121518, 0.0002319709095120632]
algorithm results returned untouched: 10 + 10 counts dicts


`monitor.export_csv("history.csv")` writes the full monitoring history; `monitor.set_baseline()` re-baselines after a known recalibration; `monitor.drift_report_all()` corrects for multiple testing across all qubits of a round.

## 11. Command line

Everything above is also available as the `quantum-canary` command:

```bash
quantum-canary simulate --architecture superconducting --qubits 3 --regime nisq
quantum-canary ibm --fake fake_fez --qubits 0,1          # local dry run
quantum-canary ibm --backend ibm_fez --qubits 13,14,2    # real hardware (saved account)
quantum-canary selftest
```

Options: `--json report.json` saves the full report, `--no-color` gives plain text, `--verbose` adds the engine's own log lines, `--probe-blocks N` sets the interleaving.

In [18]:
import os, subprocess, sys
print(subprocess.run([sys.executable, "-m", "quantum_canary", "--help"],
                     capture_output=True, text=True,
                     env={**os.environ, "NO_COLOR": "1"}).stdout)

usage: quantum-canary [-h] [--version] {ibm,simulate,selftest} ...

Quantum Canary: low-shot recovery of T1, T2, T2*, detuning and single-qubit
gate error from a fixed set of probe circuits.

positional arguments:
  {ibm,simulate,selftest}
    ibm                 run on an IBM (superconducting) backend
    simulate            run in silico on simulated qubits
    selftest            run the simulation layer's validation tests

options:
  -h, --help            show this help message and exit
  --version             show program's version number and exit



---
**What the simulated results do and do not show.** The simulator's noise includes drift, slow frequency noise, detuning, coherent gate error and stale calibration, but its basic decay and readout models follow the same equations Canary fits. Agreement with the simulator's truth therefore shows the method works under those conditions; agreement with independent reference measurements on real hardware is the test that shows it works on a device.